# 양자컴퓨팅 부트캠프 Day 1: 선형대수학과 단일 큐비트 상태벡터 NumPy 핸즈온

본 노트북은 **12시간 양자컴퓨팅 특강의 1일차 3교시 실습 교재**입니다.
복소 벡터 공간, 유니터리 연산자, Born 측정 규칙을 순수 Python과 `numpy`만으로 직접 계산하고 시뮬레이션합니다.

---

### 오늘 실습의 5대 학습 목표
1. **복소수와 계산 기저**: 복소 내적 공간 $\mathbb{C}^2$와 계산 기저 $|0\rangle, |1\rangle$를 2차원 열벡터로 정의.
2. **단일 큐비트 유니터리 게이트**: $I, X, Y, Z, H, S, T$ 게이트의 행렬 정의와 유니터리성($U^* U = I$) 전수 검증.
3. **정규화와 Born 측정 규칙**: 상태벡터의 크기 보존($\langle\psi|\psi\rangle = 1$)과 측정 확률 $P(0)=|\alpha|^2, P(1)=|\beta|^2$ 계산.
4. **위상(Phase)의 물리적 실체**: 관측 불가능한 Global Phase와 간섭을 유발하는 Relative Phase의 결정적 차이 증명.
5. **Shot Noise와 표본 수렴**: 유한한 반복 측정(Shots)에 따른 통계적 오차가 대수의 법칙($1/\sqrt{M}$)에 따라 수렴함을 확인.


In [ ]:
from __future__ import annotations

import numpy as np

# 소수점 출력 포맷 설정
np.set_printoptions(precision=4, suppress=True)

print(f'NumPy version: {np.__version__}')


## 1. 계산 기저 (Computational Basis) 상태 정의

단일 큐비트의 물리적 상태는 2차원 복소 내적 공간 $\mathbb{C}^2$의 단위 열벡터로 기술됩니다.
가장 기본이 되는 표준 정규직교기저 $|0\rangle$과 $|1\rangle$을 정의합니다:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$


In [ ]:
# 2차원 복소 열벡터 정의
KET_0 = np.array([[1.0 + 0.0j], [0.0 + 0.0j]])  # |0>
KET_1 = np.array([[0.0 + 0.0j], [1.0 + 0.0j]])  # |1>

print('|0> basis vector:')
print(KET_0)
print('\n|1> basis vector:')
print(KET_1)


## 2. 단일 큐비트 유니터리 게이트 (Unitary Gates)

양자 게이트는 닫힌 양자계의 가역적인 물리적 시간 발전을 나타내며, 선형대수학적으로 **크기를 보존하는 $2 \times 2$ 유니터리 행렬**로 표현됩니다:

$$U^* U = U U^* = I \iff U^{-1} = U^*$$

주요 단일 큐비트 게이트를 정의합니다:
- **Pauli-X (비트 반전, NOT)**: $X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$
- **Pauli-Y**: $Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$
- **Pauli-Z (위상 반전)**: $Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$
- **Hadamard (균등 중첩 생성)**: $H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$
- **Phase ($S = Z^{1/2}$)**: $S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$
- **T ($T = Z^{1/4}$)**: $T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$


In [ ]:
GATE_I = np.array([[1, 0], [0, 1]], dtype=complex)
GATE_X = np.array([[0, 1], [1, 0]], dtype=complex)
GATE_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
GATE_Z = np.array([[1, 0], [0, -1]], dtype=complex)
GATE_H = (1.0 / np.sqrt(2.0)) * np.array([[1, 1], [1, -1]], dtype=complex)
GATE_S = np.array([[1, 0], [0, 1j]], dtype=complex)
GATE_T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

def is_unitary(matrix: np.ndarray, tol: float = 1e-9) -> bool:
    dim = matrix.shape[0]
    identity = np.eye(dim, dtype=complex)
    product = matrix.conj().T @ matrix
    return bool(np.allclose(product, identity, atol=tol))

gates = {
    'I': GATE_I, 'X': GATE_X, 'Y': GATE_Y, 'Z': GATE_Z,
    'H': GATE_H, 'S': GATE_S, 'T': GATE_T
}

print('[게이트 유니터리성 검증]')
for name, g in gates.items():
    valid = is_unitary(g)
    print(f'  - Gate {name:2s}: Unitary = {valid}')
    assert valid, f'{name} 게이트는 유니터리하지 않습니다!'
print('=> 모든 기본 게이트가 벡터 크기(노름)를 보존함을 확인했습니다.')


## 3. 정규화와 Born 측정 규칙

상태벡터 $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$를 측정하면:
1. $0$이 관측될 확률: $P(0) = |\alpha|^2$
2. $1$이 관측될 확률: $P(1) = |\beta|^2$
3. 전체 확률 합: $|\alpha|^2 + |\beta|^2 = \langle\psi|\psi\rangle = 1$ (정규화 조건)


In [ ]:
def check_normalization(state: np.ndarray, tol: float = 1e-9) -> bool:
    norm_sq = np.vdot(state, state).real
    return bool(np.isclose(norm_sq, 1.0, atol=tol))

def measure_probabilities(state: np.ndarray) -> tuple[float, float]:
    p0 = float(np.abs(state[0, 0]) ** 2)
    p1 = float(np.abs(state[1, 0]) ** 2)
    return p0, p1

sample_state = np.array([[0.5], [np.sqrt(3)/2.0]], dtype=complex)

print('상태벡터 정규화 검증:', check_normalization(sample_state))
p0, p1 = measure_probabilities(sample_state)
print(f'이론적 확률: P(0) = {p0:.4f} (25%), P(1) = {p1:.4f} (75%)')


## 4. Hadamard 게이트와 균등 중첩 ($|+\rangle$)

$|0\rangle$ 상태에 Hadamard 게이트 $H$를 적용하면 0과 1이 정확히 50:50으로 관측되는 균등 중첩 상태 $|+\rangle$가 탄생합니다:

$$|+\rangle = H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = \begin{pmatrix} 1/\sqrt{2} \\ 1/\sqrt{2} \end{pmatrix}$$


In [ ]:
plus_state = GATE_H @ KET_0
print('|+> 상태벡터:')
print(plus_state)

p0, p1 = measure_probabilities(plus_state)
print(f'|+> 측정 확률: P(0) = {p0:.4f}, P(1) = {p1:.4f}')

recovered_zero = GATE_H @ plus_state
print('\nH @ |+> = H^2 |0> 복원 여부:', np.allclose(recovered_zero, KET_0))


## 5. [위상의 수학적 실체] 상대 위상(Relative Phase)과 양자 간섭

큐비트의 상태에서 **위상차(^{i\phi}$)**가 실제로 어떻게 계산에 개입하는지 관찰합니다.

- **상대 위상 (Relative Phase)**: $|\psi\rangle = \frac{|0\rangle + e^{i\phi}|1\rangle}{\sqrt{2}}$
  - 단순 $-기저로 측정하면 $|1/\sqrt{2}|^2 = 0.5$, $|e^{i\phi}/\sqrt{2}|^2 = 0.5 $\phi$ 값과 무관하게 50:50으로 보입니다.
  - 하지만 **Hadamard 게이트를 통과시키는 순간, 위상에 따른 보강/상쇄 간섭이 일어나 전혀 다른 비트로 분리**됩니다!


In [ ]:
# 1. 상대 위상이 0인 |+> 상태와 pi인 |-> 상태
psi_plus = (1.0 / np.sqrt(2.0)) * np.array([[1.0], [1.0]])   # (|0> + |1>) / sqrt(2)
psi_minus = (1.0 / np.sqrt(2.0)) * np.array([[1.0], [-1.0]]) # (|0> - |1>) / sqrt(2)

print('[실험 1: 단순 Z-기저 측정 확률]')
print('  - |+> 상태 측정 확률: ', measure_probabilities(psi_plus))
print('  - |-> 상태 측정 확률: ', measure_probabilities(psi_minus))
print('  -> 두 상태 모두 (0.5, 0.5)로 겉보기 확률은 완전히 동일합니다.')

print('\n[실험 2: Hadamard 게이트 통과 후 간섭 결과]')
h_plus = GATE_H @ psi_plus
h_minus = GATE_H @ psi_minus

print('  - H @ |+> 확률: ', np.round(measure_probabilities(h_plus), 4), ' (보강 간섭 -> |0> 100%)')
print('  - H @ |-> 확률: ', np.round(measure_probabilities(h_minus), 4), ' (상쇄 간섭 -> |1> 100%)')
print('  -> 상대 위상(부호)의 차이가 Hadamard를 만나 100% 확정된 결과의 차이로 드러납니다!')


## 6. 유한한 측정과 통계적 수렴 (Shot Noise 분석)

실제 양자 컴퓨터에서는 내부의 복소 진폭을 들여다볼 수 없으며, 오직 $M$번의 유한한 반복 측정(Shots)을 통해서만 고전 비트 빈도를 얻을 수 있습니다.
대수의 법칙에 따라, 관측 빈도와 이론적 확률의 오차는 $1/\sqrt{M}$에 비례하여 줄어듭니다.


In [ ]:
def sample_measurements(state: np.ndarray, shots: int = 1000) -> dict[str, int]:
    p0, p1 = measure_probabilities(state)
    outcomes = np.random.choice(['0', '1'], size=shots, p=[p0, p1])
    count_0 = int(np.sum(outcomes == '0'))
    return {'0': count_0, '1': shots - count_0}

print('[Shot 수에 따른 관측 비율 및 오차 수렴]')
true_p0 = 0.5
np.random.seed(42)

for shots in [10, 100, 1000, 10000, 100000]:
    counts = sample_measurements(plus_state, shots=shots)
    emp_p0 = counts['0'] / shots
    error = abs(emp_p0 - true_p0)
    print(f'  - Shots {shots:7d}: counts={counts} -> 관측 비율={emp_p0:.5f} (오차={error:.5f})')


---
### Day 1 실습 정리 과제 (Self-Study)
1. $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$ 상태벡터를 정의하고, $H|-\rangle$를 계산하여 $|1\rangle$이 나오는지 확인해 보세요.
2. $X, Y, Z$ 게이트의 고유값(Eigenvalues)과 고유벡터(Eigenvectors)를 `np.linalg.eig`로 구하고, 모든 고유값이 $\pm 1$인지 확인해 보세요.
